<a href="https://colab.research.google.com/github/nadilHesara/langid_experiments/blob/main/LangID.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Data Loading

###sinhala paali parallel corpus data

In [22]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

path = "/content/drive/MyDrive/SSLI/raw_data/sinhala_paali_parallel_corpus/output.tsv"

df = pd.read_csv(path, sep="\t")
print(df.shape)
print(df.columns.tolist())
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(28395, 2)
['pali_text', 'sinhala_text']


,pali_text,sinhala_text
0,එවං මෙ සුතං - එකං සමයං භගවා අන්තරා ච රාජගහං අන...,මා විසින් මෙසේ අසනලදී. (නොහොත් මගේ ඇසීම මෙලෙසය...
1,අථ ඛො භගවා අම්බලට්ඨිකායං රාජාගාරකෙ එකරත්තිවාසං...,එවිට භාග්‍යවතුන් වහන්සේ අම්බලට්ඨිකා උයනෙහි රජු...
2,අථ ඛො සම්බහුලානං භික්ඛූනං රත්තියා පච්චූසසමයං ප...,එකල රෑ පාන්දරින් මණ්ඩලමාල නම් රැස්වීම් ශාලාවෙහ...
3,අථ ඛො භගවා තෙසං භික්ඛූනං ඉමං සඞ්ඛියධම්මං විදිත...,එකල්හි භාග්‍යවතුන් වහන්සේ භික්ෂූන්ගේ මේ කථාව ද...
4,"‘‘මමං වා, භික්ඛවෙ, පරෙ අවණ්ණං භාසෙය්‍යුං, ධම්ම...","“මහණෙනි, අනුන් මාගේ හෝ නුගුණ කියද්ද, ධර්මය ගැන..."


###doc data

####paali

In [27]:
import os

folder = "/content/drive/.shortcut-targets-by-id/1gPCbEcE9ztxyuWFqpVtSSNpel4wklH_g/SSLI/raw_data/doc/paali"

txt_files = [f for f in os.listdir(folder) if f.endswith(".txt")]
print(f"Found {len(txt_files)} files:")
for f in txt_files:
    print(f)

Found 15 files:
බුද්ධ සික්ඛා.txt
අධිමාස දීපනය.txt
මහාවංස ටීකා.txt
සුමංගල විලාසිනි නාමා දීඝනිකාය අට්ඨකථා.txt
සාදුපත්ථපුරණී.txt
සාදුජනපපසාදිනී.txt
සමෙමාහනාසිනී.txt
මහාරූපසිද්ධි.txt
මොග්ගලලායනව්_යාකරණං.txt
මාහාවංසො (තුතියෝ භාගෝ).txt
අධිකරණ විභාග සංඝෝ.txt
ප්_රපංච සුදනීනාම මජ්ජිම නිකාය අට්ඨකථා හෙවත් මැදුම් සඟි අටුවාව.txt
දීපවංසෝ.txt
ධම්මපදට්ඨ කථා - දුතියො භාගො.txt
ජාතකට්ඨකථා.txt


In [29]:
import re, pandas as pd

label_map = {
    "අධිමාස": "pali", "කුදුසික": "pali", "ටීකා": "pali",
    "සුමංගල": "pali", "සාදුපත්ථ": "pali", "සාදුජනප": "pali",
    "සමෙමාහන": "pali", "බුද්ධ සික්ඛා": "pali", "මහාරූපසිද්ධි": "pali",
    "ග්ගලලායන": "pali", "අධිකරණ": "pali", "මාහාවංසො": "pali",
    "ප්_රපංච": "pali", "ප්රපංච": "pali",
    "පබ්බතූපම": "pali_then_sinhala",
    "ධම්මපදට්ඨ": "pali", "දීපවංසෝ": "pali", "ජාතකට්ඨකථා": "pali",
    "කවච": "sanskrit_from_line40",
    "චරියා": "EXCLUDE_COPYRIGHT",
}

import unicodedata

def normalize(s):
    return unicodedata.normalize("NFC", s)

def find_label(filename):
    filename_norm = normalize(filename)
    for key, val in label_map.items():
        if normalize(key) in filename_norm:
            return val
    return "UNKNOWN"

rows = []
skipped = []

for fname in txt_files:
    label = find_label(fname)
    book_title = fname.replace(".txt", "")

    if label in ("EXCLUDE_COPYRIGHT", "UNKNOWN"):
        skipped.append((fname, label))
        continue

    with open(os.path.join(folder, fname), encoding="utf-8") as f:
        text = f.read()

    lines = [l.strip() for l in text.split("\n")]
    lines = [l for l in lines if len(l) > 5 and not re.fullmatch(r"[\d\.\s]+", l)]

    if label == "pali_then_sinhala":
        skipped.append((fname, "needs manual pali/sinhala split"))
        continue

    if label == "sanskrit_from_line40":
        lines = lines[39:]
        label = "sanskrit"

    for sent in lines:
        rows.append({
            "text": sent,
            "label": label,
            "source": "SiDiaC-v2",
            "subcorpus": book_title,
            "group_id": f"sidiac2_{book_title}"
        })

sidiac_df = pd.DataFrame(rows)
print(sidiac_df.shape)
print(sidiac_df["label"].value_counts())
print("\nSkipped/flagged:")
for s in skipped:
    print(s)

(2929, 5)
label
pali    2929
Name: count, dtype: int64

Skipped/flagged:


####sanskrit

In [30]:
!find /content/drive -type d -iname "*sasnkrit*" 2>/dev/null

/content/drive/.shortcut-targets-by-id/1gPCbEcE9ztxyuWFqpVtSSNpel4wklH_g/SSLI/raw_data/doc/sasnkrit


In [31]:
import os

sanskrit_folder = "/content/drive/.shortcut-targets-by-id/1gPCbEcE9ztxyuWFqpVtSSNpel4wklH_g/SSLI/raw_data/doc/sasnkrit"
print(os.listdir(sanskrit_folder))

['කවච සංග්_රහය හෙවත් ආදිත්_යාදි නව ග්_රහයන් පිළිබඳ කවචයන් සහ දේවි කවචය ආදි කවචයන්.txt']


In [32]:
import re

fname = os.listdir(sanskrit_folder)[0]  # only one file in there
with open(os.path.join(sanskrit_folder, fname), encoding="utf-8") as f:
    text = f.read()

lines = [l.strip() for l in text.split("\n")]
lines = lines[39:]  # skip first 39 lines (0-indexed), keep line 40 onwards
lines = [l for l in lines if len(l) > 5 and not re.fullmatch(r"[\d\.\s]+", l)]

sanskrit_rows = pd.DataFrame([{
    "text": sent,
    "label": "sanskrit",
    "source": "SiDiaC-v2",
    "subcorpus": fname.replace(".txt", ""),
    "group_id": f"sidiac2_{fname.replace('.txt', '')}"
} for sent in lines])

print(sanskrit_rows.shape)
sanskrit_rows.head()

(101, 5)


,text,label,source,subcorpus,group_id
0,නව ග්‍රහාණං කවචං වක්ෂෙහං ශ්‍රැණු සුව්‍රත,sanskrit,SiDiaC-v2,කවච සංග්_රහය හෙවත් ආදිත්_යාදි නව ග්_රහයන් පිළි...,sidiac2_කවච සංග්_රහය හෙවත් ආදිත්_යාදි නව ග්_රහ...
1,"සර්ව සම්පත් ප්‍රදංචෛව සර්වරොග හර පරම්,",sanskrit,SiDiaC-v2,කවච සංග්_රහය හෙවත් ආදිත්_යාදි නව ග්_රහයන් පිළි...,sidiac2_කවච සංග්_රහය හෙවත් ආදිත්_යාදි නව ග්_රහ...
2,සර්ව ශත්‍රැක්ෂය චෛව භුක්ති මුක්ති ප්‍රදංශුභං,sanskrit,SiDiaC-v2,කවච සංග්_රහය හෙවත් ආදිත්_යාදි නව ග්_රහයන් පිළි...,sidiac2_කවච සංග්_රහය හෙවත් ආදිත්_යාදි නව ග්_රහ...
3,ඒකාන්ත සුද්ධ දෙශෙතු සුඛාසීන උදං මුඛ්‍ය,sanskrit,SiDiaC-v2,කවච සංග්_රහය හෙවත් ආදිත්_යාදි නව ග්_රහයන් පිළි...,sidiac2_කවච සංග්_රහය හෙවත් ආදිත්_යාදි නව ග්_රහ...
4,රීෂ්ටාදීනාසතංකෘත්වා තත්තන්ධ්‍යා නෛශවතොෂයෙත්,sanskrit,SiDiaC-v2,කවච සංග්_රහය හෙවත් ආදිත්_යාදි නව ග්_රහයන් පිළි...,sidiac2_කවච සංග්_රහය හෙවත් ආදිත්_යාදි නව ග්_රහ...
